In [0]:
print('DATABRICKS')

DATABRICKS


In [0]:
spark.conf.set(
  "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
  "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx=="
)

In [0]:
df_stores=spark.read.parquet("abfss://retails@storeb1.dfs.core.windows.net/dbo.stores.parquet")
df_stores.show()

+--------+--------------------+------------+
|store_id|          store_name|    location|
+--------+--------------------+------------+
|       1|     City Mall Store|         UAE|
|       2|   High Street Store|Saudi Arabia|
|       3|   Tech World Outlet|       Qatar|
|       4|Cairo Festival Ci...|       Egypt|
|       5|          Mega Plaza|      Kuwait|
+--------+--------------------+------------+



In [0]:
df_products=spark.read.parquet("abfss://retails@storeb1.dfs.core.windows.net/dbo.products.parquet")
df_products.show()

+----------+-----------------+-----------+-----+
|product_id|     product_name|   category|price|
+----------+-----------------+-----------+-----+
|         1|   Wireless Mouse|Electronics|  800|
|         2|Bluetooth Speaker|Electronics| 1200|
|         3|         Yoga Mat|    Fitness|  499|
|         4|     Laptop Stand|Accessories|  999|
|         5|     Notebook Set| Stationery|  149|
|         6|     Water Bottle|    Fitness|  299|
|         7|       Smartwatch|Electronics| 4999|
|         8|   Desk Organizer|Accessories|  399|
|         9|     Dumbbell Set|    Fitness| 1999|
|        10|   Pen Drive 32GB|Electronics|  599|
+----------+-----------------+-----------+-----+



In [0]:
df_transactions=spark.read.parquet("abfss://retails@storeb1.dfs.core.windows.net/dbo.transactions.parquet")
df_transactions.show()

+--------------+-----------+----------+--------+--------+----------------+
|transaction_id|customer_id|product_id|store_id|quantity|transaction_date|
+--------------+-----------+----------+--------+--------+----------------+
|            31|        101|         1|       1|       3|      2025-04-01|
|            32|        102|         2|       2|       2|      2025-04-03|
|            33|        103|         3|       3|       1|      2025-04-05|
|            34|        104|         4|       4|       5|      2025-04-07|
|            35|        105|         5|       5|       2|      2025-04-09|
|            36|        106|         6|       1|       4|      2025-04-11|
|            37|        107|         7|       2|       1|      2025-04-13|
|            38|        108|         8|       3|       3|      2025-04-15|
|            39|        109|         9|       4|       2|      2025-04-17|
|            40|        110|        10|       5|       5|      2025-04-19|
|            41|        1

In [0]:
df_customers=spark.read.parquet("abfss://retails@storeb1.dfs.core.windows.net/MohammedHameds/Retails_Project/refs/heads/main/dataset/customers.parquet")
df_customers.show()

+-----------+----------------+--------------------+------------+-----------------+
|customer_id|       full_name|               email|     country|registration_date|
+-----------+----------------+--------------------+------------+-----------------+
|        101|    Ahmed Khaled|ahmed.khaled1@gma...|       Egypt|       2025-10-16|
|        102| Sara Al Mansour|sara.almansour2@o...|Saudi Arabia|       2025-10-18|
|        103|    Layla Kazemi|layla.kazemi3@yah...|         UAE|       2025-10-19|
|        104|     Omar Farouk|omar.farouk4@gmai...|       Egypt|       2025-10-17|
|        105|Fatima Al Rashid|fatima.alrashid5@...|Saudi Arabia|       2025-10-15|
|        106|   Yousef Nasser|yousef.nasser6@gm...|         UAE|       2025-10-16|
|        107|     Mona Hossam|mona.hossam7@yaho...|       Egypt|       2025-10-18|
|        108|  Hassan Al Saud|hassan.alsaud8@ou...|Saudi Arabia|       2025-10-19|
|        109|     Ayesha Khan|ayesha.khan9@gmai...|         UAE|       2025-10-15|
|   

In [0]:
df_transactions.write.mode("overwrite").saveAsTable("retails_bisho.sales.transactions_bronze")
df_products.write.mode("overwrite").saveAsTable("retails_bisho.sales.products_bronze")
df_stores.write.mode("overwrite").saveAsTable("retails_bisho.sales.stores_bronze")
df_customers.write.mode("overwrite").saveAsTable("retails_bisho.sales.customers_bronze")

In [0]:
df_transactions.printSchema()
df_products.printSchema()
df_stores.printSchema()
df_customers.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_date: date (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)

root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- location: string (nullable = true)

root
 |-- customer_id: long (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)



In [0]:
from pyspark.sql.functions import *


In [0]:
df_customers=df_customers.select(
    col("customer_id").cast("int"),
    col("full_name"),
    col("email"),
    col("country"),
    col("registration_date").cast("date")
)
df_customers.printSchema()


root
 |-- customer_id: integer (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)



In [0]:
# ONE_TABLE Structure
df_retails=df_transactions.join(df_customers,"customer_id").join(df_products,"product_id").join(df_stores,"store_id").withColumn("total_amount", col("quantity")*col("price"))


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5677439152207249>, line 2
      1 # ONE_TABLE Structure
----> 2 df_retails=df_transactions.join(df_customers,"customer_id").join(df_products,"product_id").join(df_stores,"store_id").withColumn("total_amount", col("quantity")*col("price"))

NameError: name 'df_transactions' is not defined

In [0]:
df_retails.write.mode("overwrite").saveAsTable("retails_bisho.sales.retails_silver")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5677439152207246>, line 1
----> 1 df_retails.write.mode("overwrite").saveAsTable("retails_bisho.sales.retails_silver")

NameError: name 'df_retails' is not defined

In [0]:
display(df_retails)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5677439152207245>, line 1
----> 1 display(df_retails)

NameError: name 'df_retails' is not defined

In [0]:

df_retails=spark.sql("SELECT * FROM retails_bisho.sales.retails_silver")

+--------+----------+-----------+--------------+--------+----------------+----------------+--------------------+------------+-----------------+-----------------+-----------+-----+--------------------+------------+------------+
|store_id|product_id|customer_id|transaction_id|quantity|transaction_date|       full_name|               email|     country|registration_date|     product_name|   category|price|          store_name|    location|total_amount|
+--------+----------+-----------+--------------+--------+----------------+----------------+--------------------+------------+-----------------+-----------------+-----------+-----+--------------------+------------+------------+
|       3|         8|        101|            58|       3|      2025-05-25|    Ahmed Khaled|ahmed.khaled1@gma...|       Egypt|       2025-10-16|   Desk Organizer|Accessories|  399|   Tech World Outlet|       Qatar|        1197|
|       4|         9|        102|            59|       5|      2025-05-27| Sara Al Mansour|s

In [0]:
%sql
select * from retails_bisho.sales.stores_bronze


store_id,store_name,location
1,City Mall Store,UAE
2,High Street Store,Saudi Arabia
3,Tech World Outlet,Qatar
4,Cairo Festival City Mall,Egypt
5,Mega Plaza,Kuwait


Then use sql(tables) or spark(dataframes) to extract KPI for Gold  

In [0]:
%sql
--INSERT INTO retails_bisho.sales.top_products_gold
SELECT product_name , sum(total_amount) as total_amount
FROM retails_bisho.sales.retails_silver
GROUP BY product_name
ORDER BY total_amount DESC

product_name,total_amount
Smartwatch,69986
Dumbbell Set,33983
Laptop Stand,21978
Bluetooth Speaker,18000
Wireless Mouse,9600
Desk Organizer,6783
Pen Drive 32GB,6589
Yoga Mat,4990
Water Bottle,4186
Notebook Set,1490


In [0]:
# KPI 1 (TOP PRODUCTS)
from pyspark.sql import functions as F
df_top_products=df_retails.groupby("product_name").agg(F.sum("total_amount").alias("total_amount")).orderBy(F.col("total_amount").desc())


In [0]:
df_top_products.write.mode("overwrite").saveAsTable("retails_bisho.sales.top_products_gold")

In [0]:
# KPI 2 (TOP CATEGORIES)
from pyspark.sql import functions as F
df_top_categories=df_retails.groupby("category").agg(F.sum("total_amount").alias("total_amount")).orderBy(F.col("total_amount").desc())

In [0]:
df_top_categories.write.mode("overwrite").saveAsTable("retails_bisho.sales.top_categories_gold")

In [0]:
spark.sql(
    """
    DROP TABLE retails_bisho.sales.top_countries_gold
    """
)


DataFrame[]

In [0]:
from pyspark.sql import functions as F
df_top_countries=df_retails.groupby("country").agg(F.sum("total_amount").alias("total_amount")).orderBy(F.col("total_amount").desc())

In [0]:
df_top_countries.write.mode("overwrite").saveAsTable("retails_bisho.sales.top_countries_gold")

In [0]:
spark.sql (
    """
    DROP TABLE retails_bisho.sales.top_stores_gold
    """
)

DataFrame[]

In [0]:
#KPI 4 (TOP STORES)
from pyspark.sql import functions as F
df_top_stores=df_retails.groupby("store_name").agg(F.sum("total_amount").alias("total_amount")).orderBy(F.col("total_amount").desc())

In [0]:
df_top_stores.show()

+--------------------+------------+
|          store_name|total_amount|
+--------------------+------------+
|   High Street Store|       87986|
|Cairo Festival Ci...|       55961|
|     City Mall Store|       13786|
|   Tech World Outlet|       11773|
|          Mega Plaza|        8079|
+--------------------+------------+



In [0]:
df_top_stores.write.mode("overwrite").saveAsTable("retails_bisho.sales.top_stores_gold")

In [0]:
%sql
select * from retails_bisho.sales.top_stores_gold

store_name,total_amount
High Street Store,87986
Cairo Festival City Mall,55961
City Mall Store,13786
Tech World Outlet,11773
Mega Plaza,8079


In [0]:
# KPI 5(TOP LOCATIONS)
from pyspark.sql import functions as F
df_top_locations=df_retails.groupby("location").agg(F.sum("total_amount").alias("total_amount")).orderBy(F.col("total_amount").desc())

In [0]:
df_top_locations.write.mode("overwrite").saveAsTable("retails_bisho.sales.top_locations_gold")

In [0]:
import pyspark.sql.functions as sf
df_retails.select("transaction_date", sf.year('transaction_date')).show()
df_retails=df_retails.Withcolumn("year",sf.year('transaction_date'))


+----------------+------------------------+----------------------+
|transaction_date|typeof(transaction_date)|year(transaction_date)|
+----------------+------------------------+----------------------+
|      2025-04-01|                    date|                  2025|
|      2025-04-03|                    date|                  2025|
|      2025-04-05|                    date|                  2025|
|      2025-04-07|                    date|                  2025|
|      2025-04-09|                    date|                  2025|
|      2025-04-11|                    date|                  2025|
|      2025-04-13|                    date|                  2025|
|      2025-04-15|                    date|                  2025|
|      2025-04-17|                    date|                  2025|
|      2025-04-19|                    date|                  2025|
|      2025-04-21|                    date|                  2025|
|      2025-04-23|                    date|                  2

In [0]:
# KPI 5(TOP years)
from pyspark.sql import functions as F
df_top_years = df_retails.groupby(F.year('transaction_date').alias('year')).agg(F.sum('total_amount').alias('total_amount')).orderBy(F.col('total_amount').desc())

In [0]:
df_top_years.show()
df_top_years.write.mode("overwrite").saveAsTable("retails_bisho.sales.top_years_gold")

+----+------------+
|year|total_amount|
+----+------------+
|2025|      177585|
+----+------------+



In [0]:
# KPI 6(TOP months)
from pyspark.sql import functions as F
df_top_months = df_retails.groupBy(F.year('transaction_date').alias('year'), F.month('transaction_date').alias('month')).agg(F.sum('total_amount').alias('total_amount')).orderBy(F.col('total_amount').desc())


In [0]:
df_top_months.show()
df_top_months.write.mode("overwrite").saveAsTable("retails_bisho.sales.top_months_gold")

+----+-----+------------+
|year|month|total_amount|
+----+-----+------------+
|2025|    5|       72311|
|2025|    6|       50165|
|2025|    4|       34721|
|2025|    7|       20388|
+----+-----+------------+



In [0]:
# KPI 7 (customeres)
from pyspark.sql import functions as F
df_customers = df_retails.groupBy('customer_id').agg(F.sum('total_amount').alias('total_amount')).orderBy(F.col('total_amount').desc())
df_customers.show()
df_customers.write.mode("overwrite").saveAsTable("retails_bisho.sales.customers_gold")

+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|        117|       28991|
|        110|       17992|
|        120|       16195|
|        112|       14795|
|        102|       12395|
|        127|        9998|
|        107|        9994|
|        119|        8594|
|        122|        6799|
|        104|        6595|
|        114|        5397|
|        124|        4995|
|        109|        4596|
|        121|        3996|
|        105|        3898|
|        101|        3597|
|        106|        3192|
|        115|        2549|
|        111|        2396|
|        123|        2196|
+-----------+------------+
only showing top 20 rows


In [0]:
# KPI 7 (customeres_country)
from pyspark.sql import functions as F
df_geography= df_retails.groupBy('country').agg(F.count('customer_id').alias('total'))
df_geography.show()
df_geography.write.mode("overwrite").saveAsTable("retails_bisho.sales.geography_gold")

+------------+-----+
|     country|total|
+------------+-----+
|Saudi Arabia|   17|
|         UAE|   16|
|       Egypt|   17|
+------------+-----+

